# Notebook 01 - Warehouse Full-History EDA

Nguon: `EDA_CURATED_PLAN.md`, `discuss/eda-curated-implementation/` (file 01-08). Wave A: mo ta toan bo
lich su da hop nhat trong warehouse - KHONG phai causal training dataset.

**PHAI chay qua `python run_wave_a.py`** (tu thu muc `eda/`), KHONG mo notebook nay truc tiep trong
Jupyter roi bam Run All - runner moi la noi tao `analysis_dir` (fail-if-exists) va dat cac bien moi
truong `EDA_SRC_DIR`/`EDA_ANALYSIS_DIR` ma cell duoi day can. Xem `notebooks/README.md`.

Moi truy van DB deu qua `queries.run_metric()` (co kiem `output_schema`), moi ket noi la READ-ONLY
(server-enforced). Notebook nay chi orchestration/dien giai - logic that nam trong `src/wave_a.py`,
`src/queries.py`, `src/metrics.py`.

In [ ]:
import os
import sys
from pathlib import Path

# GPT review 12 eda B1: KHONG doan Path.cwd() - doc tu bien moi truong runner da dat truoc khi mo
# kernel. Fail RO RANG neu thieu, khong fallback CWD (nguon loi cu).
try:
    SRC_DIR = Path(os.environ["EDA_SRC_DIR"])
    ANALYSIS_DIR = Path(os.environ["EDA_ANALYSIS_DIR"])
except KeyError as exc:
    raise RuntimeError(
        "Thieu bien moi truong "
        + str(exc)
        + " - notebook nay PHAI duoc chay qua `python run_wave_a.py` (tu thu muc eda/), "
        "khong mo truc tiep trong Jupyter. Xem notebooks/README.md."
    ) from exc

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import artifacts
import contracts
import db
import holidays
import metrics
import protocol_schedule as ps
import queries
import wave_a

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (9, 5)

print("SRC_DIR:", SRC_DIR)
print("ANALYSIS_DIR:", ANALYSIS_DIR)
TABLES_DIR = ANALYSIS_DIR / "tables"
FIGURES_DIR = ANALYSIS_DIR / "figures"

## Thu thap toan bo du lieu (1 pham vi connection duy nhat - GPT review 12 M1)

`wave_a.collect_wave_a_data()` mo DUNG 1 `db.connect()`, chay het cac truy van can DB (qua
`queries.run_metric`), roi dong connection TRUOC khi doc file (ownership/cohort manifest, holiday CSV).
Assert non-terminal da nam TRONG ham nay (fail nhanh neu batch chua an toan).

In [ ]:
data = wave_a.collect_wave_a_data()
snapshot = data["snapshot"]
print("database:", snapshot.database, "| batch_id:", snapshot.batch_id)
print("canonicalization_git_commit:", snapshot.canonicalization_git_commit)
display(data["core_counts"])

In [ ]:
input_manifest = wave_a.build_input_manifest(
    data, notebook_source_path=SRC_DIR.parent / "notebooks" / "01_warehouse_full_history_eda.ipynb"
)
artifacts.atomic_write_json(ANALYSIS_DIR / "input_manifest.json", input_manifest)
print("input_manifest.json da ghi. code_provenance.is_dirty =", input_manifest["code_provenance"]["is_dirty"])
print("query_catalog_version:", input_manifest["query_catalog_version"],
      "| so metric:", len(input_manifest["query_catalog_metric_ids"]))

## 7.2 Source, ownership va protocol coverage

In [ ]:
display(data["ownership"])
print("tong so dong run/item/observation theo (nguon, ngay crawl):", len(data["run_item_obs_by_date"]))
display(data["run_item_obs_by_date"].head(10))

## 7.3 Crawl operations va capacity

Khong dung thoi luong run de suy chat luong gia - day la quality/capacity metric rieng (muc 7.3).

In [ ]:
run_duration = data["run_duration"]
display(run_duration.groupby("source_code")[["duration_minutes", "items_per_hour", "observations_per_hour"]].describe().T)
n_cross_day = int(run_duration["crosses_next_crawl_day"].sum())
print(f"so run keo qua ngay crawl ke tiep: {n_cross_day}/{len(run_duration)}")

fig, ax = plt.subplots()
for source, group in run_duration.groupby("source_code"):
    ax.hist(group["duration_minutes"], bins=30, alpha=0.6, label=source)
ax.set_xlabel("duration_minutes")
ax.set_title("Phan bo thoi luong run theo nguon")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "run_duration_by_source.png", dpi=120)
plt.show()

## 7.4 Hotel va check-in coverage

"Active hotel" = co it nhat 1 item DA DUOC OWN (owner_success/owner_failure) trong ngay - khong dung
`hotels.booking_status` hien tai (se viet lai lich su).

In [ ]:
active = data["active_hotel_by_date"]
display(active.groupby("source_code")["n_active_hotels"].describe())

fig, ax = plt.subplots()
for source, group in active.groupby("source_code"):
    ax.plot(group["vn_crawl_date"], group["n_active_hotels"], marker="o", markersize=3, label=source)
ax.set_ylabel("so hotel active")
ax.set_title("Active hotel theo ngay crawl (VN) va nguon")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "active_hotel_by_date.png", dpi=120)
plt.show()

## 7.5 Protocol continuity (that - replay tu ownership + cohort history manifest, GPT review 12 M5 + M1/M2 vong 2)

Ba metric TACH RIENG, khong gom chung "gap":

1. **Protocol continuity** (o day): lich EXPECTED tu chinh `ownership_manifest_20260916.json` +
   `cohort_history_20260916.json` (dung version cohort co hieu luc tai TUNG crawl_date, khong dung
   workbook hien tai), lay window MOI NGUON DOC LAP voi actual run (`planned_window()` cua manifest,
   cutoff tren = ngay VN cua `dump_taken_at`), LEFT JOIN voi item thuc te ->
   `owner_success`/`owner_failure_status_*`/`missing_source_run`/`missing_item_in_existing_run`.
   Hai loai missing (GPT review 12 M1): **`missing_source_run`** = ca ngay khong co run nao cua nguon
   do; **`missing_item_in_existing_run`** = ngay do CO run nhung item/hotel nay khong nam trong do.
2. **Canonical-series turnover** (muc 7.12 ben duoi): chi mo ta ngay observed, khong suy missing.
3. **Parser completeness**: xem `missingness_available_observations` (muc 7.9).

Item `error` co `hotel_id=NULL` (GPT review 12 M2) duoc thu resolve qua
`extract_hotel_slug(source_hotel_link)` truoc khi join - phan lon resolve duoc (link con nguyen slug,
chi la parser chua kip luu `hotel_id` truoc khi that bai) va duoc gan dung vao lich expected. Chi phan
THAT SU khong resolve duoc (link chet that/khong parse duoc slug) moi con o bang
`protocol_continuity_unattributed_errors`, kem sample `source_link_hash`.</cell id="61707889">

In [ ]:
protocol_summary = metrics.protocol_continuity(data["protocol_classified"], group_cols=("owner_source",))
display(protocol_summary)
print()
print("item loi KHONG resolve duoc hotel_id (that su unattributed, kem sample source_link_hash):")
display(data["protocol_unattributed_errors"])

## 7.6 Lead time va calendar coverage

Calendar feature CHINH dung `holiday_date = checkin_date`. Holiday CSV da validate + aggregate TRUOC
join (`holidays.py`), 1 dong DUY NHAT / `(checkin_date, city)`.

In [ ]:
price_main = data["price_main"]
price_main["lead_time_bucket"] = metrics.lead_time_bucket_series(price_main["lead_time"])

before = len(price_main)
price_main_cal = price_main.merge(data["checkin_calendar"], on=["checkin_date", "city"], how="left")
assert len(price_main_cal) == before, "holiday join lam nhan/mat dong"
print("OK - holiday join khong nhan dong:", before, "->", len(price_main_cal))

# MIN2: audit event-level deterministic (khong join vao 1.27tr observation, chi luu rieng)
data["holiday_csv"].events.to_csv(TABLES_DIR / "vn_holidays_events_audit.csv", index=False)

lead_time_dist = price_main["lead_time_bucket"].value_counts().reindex(metrics.LEAD_TIME_BUCKET_ORDER, fill_value=0)
lead_time_dist.to_csv(TABLES_DIR / "lead_time_bucket_distribution.csv")
display(lead_time_dist)

weekday_dist = pd.to_datetime(price_main["checkin_date"]).dt.day_name().value_counts()
month_dist = pd.to_datetime(price_main["checkin_date"]).dt.to_period("M").value_counts().sort_index()
weekday_dist.to_csv(TABLES_DIR / "checkin_weekday_distribution.csv")
month_dist.to_csv(TABLES_DIR / "checkin_month_distribution.csv")
display(weekday_dist)

In [ ]:
holiday_rate = price_main_cal.groupby("is_public_holiday", dropna=False).size()
tet_rate = price_main_cal.groupby("is_tet", dropna=False).size()
festival_rate = price_main_cal.groupby("is_festival_period", dropna=False).size()
print("observation theo is_public_holiday:", holiday_rate.to_dict())
print("observation theo is_tet:", tet_rate.to_dict())
print("observation theo is_festival_period:", festival_rate.to_dict())

fig, ax = plt.subplots()
lead_time_dist.plot(kind="bar", ax=ax)
ax.set_title("Phan bo observation (MAIN, khong sold-out) theo lead-time bucket")
ax.set_xlabel("lead-time bucket (ngay)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "lead_time_bucket_distribution.png", dpi=120)
plt.show()

## 7.7 Price distribution

Grain OBSERVATION (1 room option = 1 dong). Kem bang sensitivity o grain
`(hotel_id, checkin_date, vn_observation_date)`.

In [ ]:
price_dist_overall = metrics.price_distribution_stats(price_main)
price_dist_by_city = metrics.price_distribution_stats(price_main, group_cols=("city",))
price_dist_overall.to_csv(TABLES_DIR / "price_distribution_overall_main.csv", index=False)
price_dist_by_city.to_csv(TABLES_DIR / "price_distribution_by_city_main.csv", index=False)
display(price_dist_overall)
display(price_dist_by_city)

sensitivity = metrics.price_sensitivity_by_series(price_main, agg="median")
sensitivity.to_csv(TABLES_DIR / "price_sensitivity_by_series_median.csv", index=False)
print("so dong sensitivity (1/series-ngay):", len(sensitivity))

In [ ]:
# GPT review 12 eda MIN1: log-price PHAI la log10(gia) tren truc X, khong phai chi set_yscale('log')
# tren histogram gia thang thuong (2 thu khac nhau - cai truoc moi thay dung "duoi day" cua phan phoi).
log_price = np.log10(price_main["price_per_night"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(price_main["price_per_night"], bins=80)
axes[0].set_title("Histogram gia (thang thuong)")
axes[0].set_xlabel("price_per_night (VND)")
axes[1].hist(log_price, bins=80)
axes[1].set_title("Histogram log10(gia) - thay ro duoi phan phoi")
axes[1].set_xlabel("log10(price_per_night)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "price_histogram_main.png", dpi=120)
plt.show()

fig, ax = plt.subplots()
cities = sorted(price_main["city"].dropna().unique())
ax.boxplot([price_main.loc[price_main["city"] == c, "price_per_night"] for c in cities],
          tick_labels=cities, showfliers=False)
ax.set_ylabel("price_per_night (VND, khong hien outlier de doc duoc)")
ax.set_title("Phan bo gia theo thanh pho (box plot)")
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "price_box_by_city.png", dpi=120)
plt.show()

## 7.8 Availability state (item grain - GPT review 12 M2)

In [ ]:
main_items = data["main_items"]
status_report = metrics.status_present_report(main_items)
print("status co mat (0 nghia la KHONG bi bo qua):", status_report)

availability_overall = metrics.item_availability_rates(main_items)
availability_by_city = metrics.item_availability_rates(main_items, group_cols=("city",))
availability_by_checkin_month = metrics.item_availability_rates(
    main_items.assign(checkin_month=pd.to_datetime(main_items["checkin_date"]).dt.to_period("M").astype(str)),
    group_cols=("checkin_month",),
)
availability_overall.to_csv(TABLES_DIR / "item_availability_overall.csv", index=False)
availability_by_city.to_csv(TABLES_DIR / "item_availability_by_city.csv", index=False)
availability_by_checkin_month.to_csv(TABLES_DIR / "item_availability_by_checkin_month.csv", index=False)
display(availability_overall)
display(availability_by_city)

## 7.9 Missingness va parser completeness

Structural missing (sold-out khong co room payload) da bi loai qua dieu kien `is_sold_out=0` cua chinh
query - chi con "unexpected missing" tren observation thuc su available.

In [ ]:
missingness = data["missingness"]
missingness.to_csv(TABLES_DIR / "missingness_available_observations.csv", index=False)
display(missingness.pivot_table(index=["field_group", "field"], columns="source_code", values="null_rate"))

## 7.10 Full-history reference audit (GPT review 12 M3)

**BA bang.** Xem `discuss/eda-curated-implementation/06-claude-wave-a-code.md` + `08-...md` cho ly do
day du: bang "exact key" (GPT M3, dinh nghia chat) vs bang "series-exists" (tai hien so lich su
CLAUDE.md, GPT xac nhan giu lam metric turnover phu tro rieng - file 07 Q5).

In [ ]:
for name in ("ref_main", "ref_raw", "ref_series_exists_raw"):
    data[name]["lead_time_bucket"] = metrics.lead_time_bucket_series(data[name]["lead_time"])

cov_main = metrics.exact_approved_key_observation_coverage(data["ref_main"], group_cols=("lead_time_bucket",))
cov_raw = metrics.exact_approved_key_observation_coverage(data["ref_raw"], group_cols=("lead_time_bucket",))
# GPT review 12 eda file 09 muc 2: KHONG dung ham/cot "exact" cho metric series-level - ham va ten cot
# rieng (series_with_approved_reference_coverage / series_has_approved_reference) de khong the vo
# tinh dan nhan 30% series-exists thanh 30% exact-match.
cov_series_exists = metrics.series_with_approved_reference_coverage(
    data["ref_series_exists_raw"], group_cols=("lead_time_bucket",)
)
cov_main.to_csv(TABLES_DIR / "reference_exact_key_coverage_main.csv", index=False)
cov_raw.to_csv(TABLES_DIR / "reference_exact_key_coverage_raw.csv", index=False)
cov_series_exists.to_csv(TABLES_DIR / "reference_series_exists_coverage_raw.csv", index=False)

print("=== Bang 1: exact approved-key coverage, scope MAIN (metric chinh theo GPT file 07 Q5) ===")
display(cov_main)
print("=== Bang 2: exact approved-key coverage, scope RAW (phu luc) ===")
display(cov_raw)
print("=== Bang 3: series-exists coverage (series_has_approved_reference), scope RAW (metric turnover phu tro, GPT file 09 muc 2 xac nhan giu) ===")
display(cov_series_exists)

In [ ]:
item_level = data["ref_item_level"]
item_level["lead_time_bucket"] = metrics.lead_time_bucket_series(item_level["lead_time"].fillna(0).astype(int))
item_level_by_bucket = metrics.item_level_exact_reference_availability(item_level, group_cols=("lead_time_bucket",))
item_level_by_bucket.to_csv(TABLES_DIR / "reference_item_level_availability_by_lead_time.csv", index=False)
display(item_level_by_bucket)
display(data["ref_approval_by_city_month"])

## 7.11 Data quality findings

`quality_findings.csv` (`wave_a.build_quality_findings`): moi check co severity/scope/grain/count/
denominator/rate/sample_keys/likely_cause/recommended_action, ke ca check count=0 van co 1 dong.

In [ ]:
quality_findings = wave_a.build_quality_findings(data)
quality_findings.to_csv(ANALYSIS_DIR / "quality_findings.csv", index=False)
display(quality_findings[["check_id", "severity", "scope", "grain", "count", "denominator", "rate"]])

## 7.12 Readiness cho dataset/model (chi bao readiness - GPT review 12 M4)

`theoretical_horizon_pairs`: dem DUNG cap ngay quan sat cach nhau CHINH XAC K ngay cho tung series -
KHONG dung `n_observed_days >= K` (thuat toan cu sai, da sua theo fixture GPT dua ra).

In [ ]:
presence = data["series_presence"]
turnover = metrics.canonical_series_turnover(presence)
turnover.to_csv(TABLES_DIR / "canonical_series_turnover.csv", index=False)
print("so canonical series:", len(turnover))
display(turnover.describe())

pairs = metrics.theoretical_horizon_pairs(presence)
readiness = metrics.dataset_readiness_by_horizon(pairs)
readiness.to_csv(ANALYSIS_DIR / "dataset_readiness_by_horizon.csv", index=False)
display(readiness)

## Kiem tra trang thai Wave B (GPT review 12 M7 - dung dung dataset_version PASS)

In [ ]:
wave_b = data["wave_b_readiness"]
display(wave_b)
if wave_b.empty or not bool(wave_b["ready"].any()):
    print("Wave B: CHUA CO dataset_version nao vua status='pass' vua du 3 bang ml_*. "
          "Notebook 02 chi scaffold, khong publish ket qua.")
else:
    print("Wave B: da co it nhat 1 dataset_version san sang - can chay Notebook 02 rieng.")

## Ghi cac artifact con lai (report/dictionary/summary) - manifest cuoi cung do `run_wave_a.py` ghi

In [ ]:
wave_a.write_eda_summary(data, ANALYSIS_DIR)
wave_a.write_eda_report_and_dictionary(data, ANALYSIS_DIR)
print("Da ghi eda_summary.json, EDA_REPORT.md, DATA_DICTIONARY.md.")
print("\nDanh sach file da co trong", ANALYSIS_DIR, "(artifact_manifest.json se do run_wave_a.py ghi sau khi notebook chay xong):")
for p in sorted(ANALYSIS_DIR.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(ANALYSIS_DIR))